In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "orders.csv"
DATA_DIR = BASE_DIR / "data"

spark = (
    SparkSession.builder
    .appName("BronzeSilverGoldQualityPipeline")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Input path:", INPUT_PATH.resolve())

In [ ]:
schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
])

source_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(INPUT_PATH))
)

source_df.printSchema()
source_df.show(truncate=False)

source_count = source_df.count()
print("Source rows:", source_count)

In [ ]:
bronze_path = DATA_DIR / "bronze" / "orders"

(
    source_df.write
    .mode("overwrite")
    .parquet(str(bronze_path))
)

bronze_df = spark.read.parquet(str(bronze_path))
bronze_count = bronze_df.count()

assert bronze_count == source_count, (
    f"Bronze row-count mismatch: source={source_count}, bronze={bronze_count}"
)

print("Bronze rows:", bronze_count)

In [ ]:
negative_amount = F.col("amount") < 0

reason = (
    F.when(F.col("order_id").isNull(), F.lit("missing_order_id"))
     .when(F.col("customer_id").isNull(), F.lit("missing_customer_id"))
     .when(F.col("amount").isNull(), F.lit("missing_amount"))
     .when(negative_amount, F.lit("negative_amount"))
)

classified_df = bronze_df.withColumn("rejection_reason", reason)

invalid_df = classified_df.filter(F.col("rejection_reason").isNotNull())
valid_df = (
    classified_df
    .filter(F.col("rejection_reason").isNull())
    .drop("rejection_reason")
    .withColumn("status", F.upper(F.trim(F.col("status"))))
)

quarantine_path = DATA_DIR / "quarantine" / "orders"
(
    invalid_df.write
    .mode("overwrite")
    .parquet(str(quarantine_path))
)

print("Invalid rows:", invalid_df.count())
invalid_df.select("order_id", "customer_id", "amount", "rejection_reason").show(truncate=False)

In [ ]:
silver_df = valid_df.dropDuplicates(["order_id"])

# Quality gate: no required nulls remain.
required_nulls = silver_df.filter(
    F.col("order_id").isNull()
    | F.col("customer_id").isNull()
    | F.col("amount").isNull()
)
assert required_nulls.limit(1).count() == 0, "Required nulls found in Silver"

# Quality gate: no negative amounts remain.
assert silver_df.filter(F.col("amount") < 0).limit(1).count() == 0, (
    "Negative amounts found in Silver"
)

# Quality gate: order_id is unique.
duplicate_keys = (
    silver_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)
assert duplicate_keys.limit(1).count() == 0, "Duplicate order IDs found in Silver"

silver_path = DATA_DIR / "silver" / "orders"
(
    silver_df.write
    .mode("overwrite")
    .parquet(str(silver_path))
)

silver_df = spark.read.parquet(str(silver_path))
silver_count = silver_df.count()

print("Silver rows:", silver_count)
silver_df.orderBy("order_id").show(truncate=False)

In [ ]:
gold_df = (
    silver_df
    .groupBy("status")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("amount").alias("total_amount")
    )
)

gold_path = DATA_DIR / "gold" / "orders_by_status"
(
    gold_df.write
    .mode("overwrite")
    .parquet(str(gold_path))
)

gold_df = spark.read.parquet(str(gold_path))
gold_df.orderBy("status").show(truncate=False)

In [ ]:
gold_order_total = gold_df.agg(F.sum("total_orders").alias("n")).first()["n"]
quarantine_count = invalid_df.count()

metrics = {
    "source_rows": source_count,
    "bronze_rows": bronze_count,
    "silver_rows": silver_count,
    "quarantine_rows": quarantine_count,
    "gold_groups": gold_df.count(),
    "gold_order_total": gold_order_total,
}

print("Pipeline metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

assert source_count == bronze_count, "Source/Bronze reconciliation failed"
assert source_count == silver_count + quarantine_count + 1, (
    "Expected source = Silver + quarantine + one duplicate source row"
)
assert gold_order_total == silver_count, "Gold order total does not reconcile to Silver"

# Fixture-specific expected results
assert silver_count == 4, f"Expected 4 Silver rows, got {silver_count}"
assert quarantine_count == 2, f"Expected 2 quarantined rows, got {quarantine_count}"

print("All quality gates passed.")